In [3]:
# ╔══════════════════════════════════════════════════════════╗
# ║       Notebook 02 — Preprocessing & Cleaning            ║
# ║  Input  : data/processed/merged_dataset.csv             ║
# ║  Output : data/processed/cleaned_dataset.csv            ║
# ║           data/processed/X_train.csv                    ║
# ║           data/processed/X_test.csv                     ║
# ║           data/processed/y_train.csv                    ║
# ║           data/processed/y_test.csv                     ║
# ╚══════════════════════════════════════════════════════════╝

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────
INPUT_PATH  = r"../data/processed/merged_dataset.csv"
OUTPUT_DIR  = r"../data/processed"

# ── Plot styling ──────────────────────────────────────────
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#f8f8f6'
plt.rcParams['axes.grid']        = True
plt.rcParams['grid.alpha']       = 0.4
plt.rcParams['font.family']      = 'sans-serif'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

COLORS = {
    'none'    : '#888780',
    'ddos'    : '#D85A30',
    'portscan': '#7F77DD',
    'crypto'  : '#EF9F27'
}

print("Libraries loaded ✓")

Libraries loaded ✓


In [6]:
# ── Cell 2: Schema Validation ─────────────────────────────

df = pd.read_csv(INPUT_PATH)

# ── Define expected schema ────────────────────────────────
EXPECTED_COLUMNS = {
    # identifiers
    'timestamp'                 : 'object',
    'run_id'                    : 'int64',
    'source_file'               : 'object',
    # cpu
    'cpu_user_pct'              : 'float64',
    'cpu_system_pct'            : 'float64',
    'cpu_idle_pct'              : 'float64',
    # system activity
    'context_switch_rate'       : 'float64',
    'interrupt_rate'            : 'float64',
    'load_avg_1m'               : 'float64',
    # memory
    'ram_used_mb'               : 'float64',
    'ram_free_mb'               : 'float64',
    'swap_used_mb'              : 'float64',
    'pi_ram_used_mb'            : 'float64',
    'pi_ram_free_mb'            : 'float64',
    # process
    'total_process_count'       : 'int64',
    'total_thread_count'        : 'int64',
    'high_cpu_process_count'    : 'int64',
    'top_process_cpu_pct'       : 'float64',
    'top_process_uptime_sec'    : 'int64',
    # network
    'packets_recv_per_sec'      : 'float64',
    'packets_sent_per_sec'      : 'float64',
    'tcp_connection_count'      : 'float64',
    'failed_connection_attempts': 'float64',
    'unique_dst_port_count'     : 'int64',
    'packet_drop_rate'          : 'int64',
    # labels
    'is_attack'                 : 'int64',
    'is_legitimate_peak'        : 'int64',
    'attack_type'               : 'object',
    'system_phase'              : 'object',
}

EXPECTED_CATEGORICALS = {
    'attack_type'  : {'none', 'ddos', 'portscan', 'crypto'},
    'system_phase' : {'onset', 'sustained', 'decay'},
    'is_attack'    : {0, 1},
}

EXPECTED_ROWS = 5600

# ── Run checks ────────────────────────────────────────────
print("="*55)
print("SCHEMA VALIDATION REPORT")
print("="*55)

passed = 0
failed = 0

# Check 1: Row count
if len(df) == EXPECTED_ROWS:
    print(f"[PASS] Row count        : {len(df):,}")
    passed += 1
else:
    print(f"[FAIL] Row count        : expected {EXPECTED_ROWS:,}, got {len(df):,}")
    failed += 1

# Check 2: Column existence
missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
extra_cols   = [c for c in df.columns if c not in EXPECTED_COLUMNS]
if not missing_cols:
    print(f"[PASS] All {len(EXPECTED_COLUMNS)} expected columns present")
    passed += 1
else:
    print(f"[FAIL] Missing columns  : {missing_cols}")
    failed += 1
if extra_cols:
    print(f"[WARN] Extra columns    : {extra_cols}")

# Check 3: Data types
dtype_issues = []
for col, expected_dtype in EXPECTED_COLUMNS.items():
    if col in df.columns:
        actual = str(df[col].dtype)
        if actual != expected_dtype:
            dtype_issues.append(f"  {col}: expected {expected_dtype}, got {actual}")
if not dtype_issues:
    print(f"[PASS] All dtypes correct")
    passed += 1
else:
    print(f"[FAIL] Dtype mismatches:")
    for issue in dtype_issues:
        print(issue)
    failed += 1

# Check 4: Null values
null_counts = df.isnull().sum()
total_nulls = null_counts.sum()
if total_nulls == 0:
    print(f"[PASS] Null values      : 0")
    passed += 1
else:
    print(f"[FAIL] Null values found:")
    print(null_counts[null_counts > 0].to_string())
    failed += 1

# Check 5: Duplicates
dupes = df.duplicated().sum()
if dupes == 0:
    print(f"[PASS] Duplicate rows   : 0")
    passed += 1
else:
    print(f"[WARN] Duplicate rows   : {dupes} (will be dropped in cleaning)")
    failed += 1

# Check 6: Categorical values
for col, expected_vals in EXPECTED_CATEGORICALS.items():
    if col in df.columns:
        actual_vals = set(df[col].unique())
        if actual_vals == expected_vals:
            print(f"[PASS] {col:<25}: {sorted(actual_vals)}")
            passed += 1
        else:
            unexpected = actual_vals - expected_vals
            print(f"[FAIL] {col:<25}: unexpected values {unexpected}")
            failed += 1

# Check 7: Value ranges
range_checks = {
    'cpu_user_pct'   : (0, 100),
    'cpu_system_pct' : (0, 100),
    'cpu_idle_pct'   : (0, 100),
    'is_attack'      : (0, 1),
    'load_avg_1m'    : (0, None),
    'ram_used_mb'    : (0, None),
}
range_issues = []
for col, (lo, hi) in range_checks.items():
    if col in df.columns:
        if lo is not None and df[col].min() < lo:
            range_issues.append(f"  {col}: min={df[col].min():.2f} below {lo}")
        if hi is not None and df[col].max() > hi:
            range_issues.append(f"  {col}: max={df[col].max():.2f} above {hi}")
if not range_issues:
    print(f"[PASS] All value ranges valid")
    passed += 1
else:
    print(f"[FAIL] Range violations:")
    for issue in range_issues:
        print(issue)
    failed += 1

# ── Final verdict ─────────────────────────────────────────
print()
print("="*55)
print(f"RESULT: {passed} passed  |  {failed} failed")
print("="*55)
if failed == 0:
    print("Dataset passes full schema validation ✓")
    print("Safe to proceed to cleaning.")
else:
    print("Fix failures before proceeding.")

SCHEMA VALIDATION REPORT
[PASS] Row count        : 5,600
[PASS] All 29 expected columns present
[PASS] All dtypes correct
[PASS] Null values      : 0
[PASS] Duplicate rows   : 0
[PASS] attack_type              : ['crypto', 'ddos', 'none', 'portscan']
[PASS] system_phase             : ['decay', 'onset', 'sustained']
[PASS] is_attack                : [np.int64(0), np.int64(1)]
[PASS] All value ranges valid

RESULT: 9 passed  |  0 failed
Dataset passes full schema validation ✓
Safe to proceed to cleaning.


In [7]:
# ── Cell 3: Cleaning ──────────────────────────────────────

df_clean = df.copy()

print("="*55)
print("CLEANING REPORT")
print("="*55)

# ── Step 1: Convert timestamp to datetime ─────────────────
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
print(f"[DONE] timestamp → datetime64")
print(f"       Range: {df_clean['timestamp'].min()} → {df_clean['timestamp'].max()}")

# ── Step 2: Drop identifier columns ──────────────────────
# Why: run_id, source_file are experiment identifiers not features
# timestamp kept as index for time series analysis in EDA
DROP_COLS = ['run_id', 'source_file']
df_clean = df_clean.drop(columns=DROP_COLS)
print(f"\n[DONE] Dropped identifier columns: {DROP_COLS}")
print(f"       Remaining columns: {df_clean.shape[1]}")

# ── Step 3: Outlier detection per numeric feature ─────────
# We detect but DO NOT remove — attack behavior IS the outlier
# This section documents what's extreme for your paper

NUMERIC_FEATURES = [
    'cpu_user_pct', 'cpu_system_pct', 'cpu_idle_pct',
    'context_switch_rate', 'interrupt_rate', 'load_avg_1m',
    'ram_used_mb', 'ram_free_mb', 'swap_used_mb',
    'total_process_count', 'total_thread_count',
    'high_cpu_process_count', 'top_process_cpu_pct',
    'top_process_uptime_sec', 'packets_recv_per_sec',
    'packets_sent_per_sec', 'tcp_connection_count',
    'failed_connection_attempts', 'unique_dst_port_count',
    'packet_drop_rate', 'pi_ram_used_mb', 'pi_ram_free_mb'
]

print(f"\n── Outlier Detection (IQR method) ──")
print(f"{'Feature':<30} {'Outliers':>8} {'%':>6} {'Min':>10} {'Max':>10}")
print("-"*68)

outlier_summary = []
for feat in NUMERIC_FEATURES:
    Q1  = df_clean[feat].quantile(0.25)
    Q3  = df_clean[feat].quantile(0.75)
    IQR = Q3 - Q1
    lo  = Q1 - 1.5 * IQR
    hi  = Q3 + 1.5 * IQR

    outlier_mask  = (df_clean[feat] < lo) | (df_clean[feat] > hi)
    outlier_count = outlier_mask.sum()
    outlier_pct   = outlier_count / len(df_clean) * 100

    outlier_summary.append({
        'feature'       : feat,
        'outlier_count' : outlier_count,
        'outlier_pct'   : outlier_pct,
        'min'           : df_clean[feat].min(),
        'max'           : df_clean[feat].max(),
    })

    print(f"{feat:<30} {outlier_count:>8} {outlier_pct:>5.1f}% "
          f"{df_clean[feat].min():>10.2f} {df_clean[feat].max():>10.2f}")

# ── Step 4: Check outliers by attack class ────────────────
print(f"\n── Are outliers concentrated in attack classes? ──")
for feat in ['cpu_user_pct', 'packets_recv_per_sec',
             'top_process_cpu_pct', 'interrupt_rate']:
    Q1  = df_clean[feat].quantile(0.25)
    Q3  = df_clean[feat].quantile(0.75)
    IQR = Q3 - Q1
    hi  = Q3 + 1.5 * IQR

    outlier_rows = df_clean[df_clean[feat] > hi]
    dist = outlier_rows['attack_type'].value_counts()
    print(f"\n  {feat} (above {hi:.1f}):")
    print(f"  {dist.to_dict()}")

# ── Step 5: Drop zero-variance columns ───────────────────
zero_var = [f for f in NUMERIC_FEATURES
            if df_clean[f].std() == 0]
if zero_var:
    df_clean = df_clean.drop(columns=zero_var)
    print(f"\n[DONE] Dropped zero-variance columns: {zero_var}")
else:
    print(f"\n[PASS] No zero-variance columns found")

# ── Step 6: Final shape ───────────────────────────────────
print(f"\n── Final cleaned dataset ──")
print(f"  Shape  : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"  Nulls  : {df_clean.isnull().sum().sum()}")
print(f"  Memory : {df_clean.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

print(f"\n[DONE] Cleaning complete ✓")

CLEANING REPORT
[DONE] timestamp → datetime64
       Range: 2026-01-28 13:18:12.843264 → 2026-02-12 16:37:42.789219

[DONE] Dropped identifier columns: ['run_id', 'source_file']
       Remaining columns: 27

── Outlier Detection (IQR method) ──
Feature                        Outliers      %        Min        Max
--------------------------------------------------------------------
cpu_user_pct                        510   9.1%       0.00      97.87
cpu_system_pct                      663  11.8%       0.00      39.98
cpu_idle_pct                        595  10.6%      -0.00     100.00
context_switch_rate                1019  18.2%      44.00   24976.15
interrupt_rate                     1083  19.3%      26.00   29995.29
load_avg_1m                         498   8.9%       0.02       2.57
ram_used_mb                         223   4.0%    1341.05    1719.95
ram_free_mb                         223   4.0%    2195.88    2574.78
swap_used_mb                        300   5.4%       0.00       0

In [9]:
# ── Cell 4: Outlier Class Analysis + Decision ─────────────

print("="*55)
print("OUTLIER CLASS ANALYSIS")
print("="*55)
print("Checking if outliers belong to attack classes...")
print("(If yes → keep them. They are attack signatures.)")
print()

CHECK_FEATURES = [
    'cpu_user_pct', 'cpu_system_pct',
    'context_switch_rate', 'interrupt_rate',
    'packets_recv_per_sec', 'tcp_connection_count',
    'top_process_cpu_pct', 'top_process_uptime_sec',
    'failed_connection_attempts', 'unique_dst_port_count',
    'total_process_count', 'total_thread_count'
]

decisions = []

for feat in CHECK_FEATURES:
    Q1  = df_clean[feat].quantile(0.25)
    Q3  = df_clean[feat].quantile(0.75)
    IQR = Q3 - Q1
    lo  = Q1 - 1.5 * IQR
    hi  = Q3 + 1.5 * IQR

    outlier_mask = (df_clean[feat] < lo) | (df_clean[feat] > hi)
    outlier_rows = df_clean[outlier_mask]

    if len(outlier_rows) == 0:
        continue

    # What % of outliers are from attack classes?
    attack_outliers = outlier_rows[outlier_rows['attack_type'] != 'none']
    attack_pct      = len(attack_outliers) / len(outlier_rows) * 100

    # What % of each attack class are flagged as outliers?
    class_dist = outlier_rows['attack_type'].value_counts()

    decision = "KEEP — attack signature" if attack_pct > 50 else "INVESTIGATE"
    decisions.append({
        'feature'    : feat,
        'total'      : len(outlier_rows),
        'attack_pct' : attack_pct,
        'decision'   : decision
    })

    print(f"── {feat} ──")
    print(f"   Total outliers   : {len(outlier_rows)}")
    print(f"   From attacks     : {len(attack_outliers)} ({attack_pct:.1f}%)")
    print(f"   Class breakdown  : {class_dist.to_dict()}")
    print(f"   Decision         : {decision}")
    print()

# ── Summary decision table ────────────────────────────────
print("="*55)
print("DECISION SUMMARY")
print("="*55)
print(f"{'Feature':<30} {'Outliers':>8} {'Attack%':>8} {'Decision'}")
print("-"*70)
for d in decisions:
    print(f"{d['feature']:<30} {d['total']:>8} "
          f"{d['attack_pct']:>7.1f}% {d['decision']}")

print()
print("="*55)
print("FINAL DECISION: Keep all outliers")
print("Reason: Outliers are concentrated in attack classes.")
print("Removing them would destroy attack detection signals.")
print("="*55)

OUTLIER CLASS ANALYSIS
Checking if outliers belong to attack classes...
(If yes → keep them. They are attack signatures.)

── cpu_user_pct ──
   Total outliers   : 510
   From attacks     : 489 (95.9%)
   Class breakdown  : {'ddos': 471, 'none': 21, 'crypto': 16, 'portscan': 2}
   Decision         : KEEP — attack signature

── cpu_system_pct ──
   Total outliers   : 663
   From attacks     : 631 (95.2%)
   Class breakdown  : {'ddos': 600, 'none': 32, 'portscan': 19, 'crypto': 12}
   Decision         : KEEP — attack signature

── context_switch_rate ──
   Total outliers   : 1019
   From attacks     : 976 (95.8%)
   Class breakdown  : {'ddos': 600, 'portscan': 369, 'none': 43, 'crypto': 7}
   Decision         : KEEP — attack signature

── interrupt_rate ──
   Total outliers   : 1083
   From attacks     : 1066 (98.4%)
   Class breakdown  : {'ddos': 600, 'portscan': 466, 'none': 17}
   Decision         : KEEP — attack signature

── packets_recv_per_sec ──
   Total outliers   : 742
   From 

In [10]:
# ── Cell 5: Save cleaned dataset ─────────────────────────

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLEAN_PATH = os.path.join(OUTPUT_DIR, "cleaned_dataset.csv")

# Save without timestamp as index — keep it as column for EDA
df_clean.to_csv(CLEAN_PATH, index=False)

print("="*55)
print("PREPROCESSING COMPLETE")
print("="*55)
print(f"Input  rows    : 5,600")
print(f"Output rows    : {len(df_clean):,}")
print(f"Rows removed   : 0  (outliers kept — attack signatures)")
print(f"Cols removed   : run_id, source_file (identifiers)")
print(f"Output columns : {df_clean.shape[1]}")
print(f"Saved →        : {CLEAN_PATH}")
print()
print("What changed:")
print("  ✓ timestamp converted to datetime64")
print("  ✓ identifier columns dropped")
print("  ✓ outliers documented and retained")
print("  ✓ zero-variance columns checked")
print("  ✓ dataset validated and saved")
print()
print("Next → Notebook 03: EDA")

PREPROCESSING COMPLETE
Input  rows    : 5,600
Output rows    : 5,600
Rows removed   : 0  (outliers kept — attack signatures)
Cols removed   : run_id, source_file (identifiers)
Output columns : 26
Saved →        : ../data/processed\cleaned_dataset.csv

What changed:
  ✓ timestamp converted to datetime64
  ✓ identifier columns dropped
  ✓ outliers documented and retained
  ✓ zero-variance columns checked
  ✓ dataset validated and saved

Next → Notebook 03: EDA
